# Причинный вывод на практике: Lalonde NSW
## Часть 2 — Наблюдательный датасет: наивная оценка и смещение

**Бизнес-вопрос тот же, что в Части 1:** эффект участия в программе NSW на доход
`re78`. Но теперь мы намеренно портим дизайн — заменяем родной рандомизированный
control на людей, которые вообще не участвовали в эксперименте: общий опрос
населения CPS (Current Population Survey).

**Зачем.** В реальной продуктовой аналитике честного RCT почти никогда нет — есть
только "те, кто воспользовался фичей" и "те, кто нет", и это никогда не случайный
выбор. Эта часть имитирует именно такую ситуацию: у нас остаётся тот же treatment
(185 участников программы), но control собирается из другого источника, где
попадание в выборку никак не связано с участием в NSW.

**Эталон для сравнения** — оценка из Части 1, посчитанная на честном RCT:
**ATE = 1,794 (95% CI: [554, 3,035], SE: 633)**. Загружаем её ниже из
`benchmark.json`, а не переписываем текстом, чтобы не рассинхронизироваться с
Частью 1 при пересчёте.

## 2. Почему именно этот метод на этом шаге

На этом шаге сознательно применяется **тот же наивный diff-in-means**, что и в
Части 1 — никакого matching, весов или регрессии. Это методологически важно: если
бы мы сразу начали корректировать оценку (PSM/DML), то не увидели бы, **насколько
плохо** выглядит наивное наблюдательное сравнение само по себе, а именно это и есть
отправная точка, ради которой существуют более сложные методы.


In [ ]:
!pip install -q causaldata --break-system-packages 2>/dev/null || pip install -q causaldata

In [ ]:
import sys
sys.path.append('/content')

import pandas as pd
import numpy as np
from causaldata import nsw_mixtape, cps_mixtape

from ci_utils import smd_table, diff_in_means, load_benchmark, append_result

nsw = nsw_mixtape.load_pandas().data
cps = cps_mixtape.load_pandas().data

print(f'NSW: {nsw.shape[0]} наблюдений (experimental)')
print(f'CPS: {cps.shape[0]} наблюдений (общий опрос населения, все treat=0)')

NSW: 445 наблюдений (experimental)
CPS: 15992 наблюдений (общий опрос населения, все treat=0)


## 3. Сборка наблюдательного датасета

Берём **только treated из NSW** (185 человек, `treat=1`) — это реальные участники
программы, их данные не трогаем. Родной NSW-control (260 человек) выбрасываем
полностью — он был получен рандомизацией и в этой части не нужен, мы имитируем
ситуацию, где рандомизации не было вообще. Вместо него — весь пул CPS (`treat=0`
по построению, 15,992 человека).

In [ ]:
treated = nsw[nsw['treat'] == 1].copy()
control = cps.copy()

df = pd.concat([treated, control], ignore_index=True)

print(f'Treated (NSW): {len(treated)}')
print(f'Control (CPS): {len(control)}')
print(f'Итоговый датасет: {df.shape[0]} наблюдений')
print()
print(df['treat'].value_counts())

Treated (NSW): 185
Control (CPS): 15992
Итоговый датасет: 16177 наблюдений

treat
0    15992
1      185
Name: count, dtype: int64


## 4. Проверка баланса

В Части 1 SMD по всем ковариатам укладывался в допустимые пределы (кроме одной
пограничной) — это было прямым следствием рандомизации. Здесь рандомизации не было,
поэтому ожидаем совсем другую картину.

In [ ]:
covariates = ['age', 'educ', 'black', 'hisp', 'marr', 'nodegree', 're74', 're75']
balance = smd_table(df, 'treat', covariates)
balance.round(2)

,mean_treat,mean_control,smd
covariate,,,
age,25.82,33.23,-0.80
educ,10.35,12.03,-0.68
black,0.84,0.07,2.43
hisp,0.06,0.07,-0.05
marr,0.19,0.71,-1.23
nodegree,0.71,0.30,0.90
re74,2095.57,14016.80,-1.57
re75,1532.06,13650.80,-1.75


**Разбор таблицы:**

- `re74` (SMD −1.57) и `re75` (SMD −1.75) — участники NSW в среднем зарабатывали
  **2,096** и **1,532** за два предэкспериментальных года соответственно, против
  **14,017** и **13,651** у CPS-респондентов. Это ожидаемо: NSW набирала именно
  длительно безработных, CPS — случайный срез работающего населения.
- `age` (SMD −0.80) и `marr` (SMD −1.23) — участники NSW в среднем на 7 лет моложе
  и почти в 4 раза реже состоят в браке (19% против 71%).
- `black` (SMD +2.43) — экстремальный дисбаланс: 84% участников NSW чернокожие,
  против 7% в CPS. Это отражает демографию исходной программы, а не свойство
  причинного эффекта.

Каждая из этих ковариат — правдоподобный confounder: она связана и с тем, кто
оказался в treatment-группе (набор в NSW был не случаен), и с исходом `re78`
(возраст, брак, раса и прошлый доход сами по себе предсказывают будущий доход,
независимо от программы). Это ровно то, что определение confounding и требует —
и ровно то, чего не было в Части 1.

## 5. Наивная оценка на испорченном датасете

In [ ]:
naive = diff_in_means(df, 're78', 'treat')

print(f"ATE (diff-in-means): {naive['ate']:,.0f}")
print(f"SE: {naive['se']:,.0f}")
print(f"95% CI: [{naive['ci_low']:,.0f}, {naive['ci_high']:,.0f}]")
print(f"p-value: {naive['p_value']:.2e}")

ATE (diff-in-means): -8,498
SE: 712
95% CI: [-9,893, -7,102]
p-value: 1.07e-32


## 6. Сравнение с эталоном из Части 1

In [ ]:
benchmark = load_benchmark('benchmark.json')

bias = naive['ate'] - benchmark['ate']

print(f"Эталон (RCT, Часть 1):        {benchmark['ate']:,.0f}")
print(f"Наивная оценка (NSW+CPS):     {naive['ate']:,.0f}")
print(f"Смещение (bias):              {bias:,.0f}")
print(f"Смещение в SE эталона:        {bias / benchmark['se']:.1f} SE")

Эталон (RCT, Часть 1):        1,794
Наивная оценка (NSW+CPS):     -8,498
Смещение (bias):              -10,292
Смещение в SE эталона:        -16.3 SE


**Итог: смещение не просто заметное, а разворачивает знак эффекта.** Наивная
оценка на NSW+CPS — примерно **−8,498** (участники программы в среднем
зарабатывают на 8,498 меньше, чем control), при истинном эффекте **+1,794**.
Разница — около 10,300, это на порядок больше самого эффекта и почти в 15 раз
больше стандартной ошибки эталона.

CPS-респонденты в среднем на
порядок богаче по `re74`/`re75` ещё **до** начала программы, потому что это в целом
работающее население, а не выборка недавно безработных.

## Что дальше в этом проекте

Сохраняем наивную оценку в `results.csv` — сквозную таблицу, куда каждая следующая
часть (PSM, DML) будет добавлять свою строку, чтобы в конце проекта была одна
таблица со всеми методами против эталона.

Следующий шаг — **Часть 3: PSM/Matching** — оценка propensity score на этом же
датасете, balance-таблица до/после matching, overlap/common support, и ATT после
подбора пар, с той же логикой сравнения против эталона 1,794.

In [ ]:
append_result(naive, label='Naive diff-in-means (NSW+CPS)', path='results.csv')

,method,ate,se,ci_low,ci_high,p_value,n_treat,n_control
0,Naive diff-in-means (NSW+CPS),-8497.515625,712.020724,-9893.155036,-7101.876214,1.074814e-32,185,15992
